In [5]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
import numpy as np
from sklearn.metrics import f1_score
from tqdm import tqdm
import collections

In [6]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# Метрика Pk (Beeferman's Pk)
def calculate_pk(reference, hypothesis, k=None):
    """
    reference: список бинарных границ (1 - граница сегмента, 0 - нет)
    hypothesis: список предсказанных границ
    """
    reference = np.array(reference)
    hypothesis = np.array(hypothesis)

    n = len(reference)
    if n < 2: return 0.0


    if k is None:
        # k обычно берется как половина средней длины сегмента в эталоне
        num_boundaries = np.sum(reference)
        if num_boundaries == 0:
            k = int(n / 2)
        else:
            k = int(round(n / (2.0 * (num_boundaries + 1))))

    k = max(1, min(k, n - 1))

    def is_diff(arr, i, j):
        return 1 if np.sum(arr[i:j]) > 0 else 0

    errors = 0
    for i in range(n - k):
        ref_diff = is_diff(reference, i, i + k)
        hyp_diff = is_diff(hypothesis, i, i + k)
        if ref_diff != hyp_diff:
            errors += 1

    return errors / (n - k)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=3.0, weight=None, ignore_index=-100):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, reduction='none', ignore_index=self.ignore_index
)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()

In [ ]:
class TATSSegmenter(nn.Module):
    def __init__(self, model_name, num_classes, hidden_dropout_prob=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Многослойный MLP с Dropout и LayerNorm
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.LayerNorm(hidden_size * 4),
            nn.GELU(),
            nn.Dropout(hidden_dropout_prob),
            nn.Linear(hidden_size * 4, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(hidden_dropout_prob),
            nn.Linear(hidden_size, num_classes)
        )

        # Инициализация весов Xavier
        self._init_weights()

    def _init_weights(self):
        # Инициализация всех Linear слоёв Xavier
        for module in self.mlp:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        logits = self.mlp(sequence_output)
        return logits

In [12]:
class WikiSectionDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, label2id, max_len=512, stride=128):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len
        self.stride = stride

        self.tag2id = {}
        sorted_labels = sorted(label2id.keys())
        tags = []
        for l in sorted_labels:
            tags.extend([f"B-{l}", f"I-{l}", f"L-{l}"])
        self.tag2id = {t: i for i, t in enumerate(tags)}
        self.id2tag = {i: t for t, i in self.tag2id.items()}

        self.samples = []
        self._preprocess()

    def _preprocess(self):
        print("Preprocessing data and generating BIL tags...")
        KEY_LABEL = 'sectionLabel'
        KEY_BEGIN = 'begin'
        KEY_LENGTH = 'length'

        for row in tqdm(self.dataset):
            text = row['text']
            annotations = row['annotations']

            char_labels = [None] * len(text)
            for ann in annotations:
                try:
                    lbl = ann[KEY_LABEL]
                    start = ann[KEY_BEGIN]
                    length = ann[KEY_LENGTH]
                    end = start + length

                    for i in range(start, min(end, len(text))):
                        char_labels[i] = lbl
                except KeyError:
                    continue

            encoding = self.tokenizer(
                text,
                truncation=True,
                max_length=self.max_len,
                stride=self.stride,
                return_overflowing_tokens=True,
                return_offsets_mapping=True,
                padding="max_length"
            )

            for i, offsets in enumerate(encoding['offset_mapping']):
                input_ids = encoding['input_ids'][i]
                mask = encoding['attention_mask'][i]

                chunk_labels = []
                sent_ids = []
                current_sent_id = 0

                for start, end in offsets:
                    if start == end: # Special tokens
                        chunk_labels.append(None)
                        sent_ids.append(-1)
                        continue

                    mid = (start + end) // 2
                    lbl = char_labels[mid] if mid < len(char_labels) else None
                    chunk_labels.append(lbl)
                    
                    sent_ids.append(current_sent_id)
                    token_str = text[start:end]
                    if '.' in token_str:
                        current_sent_id += 1

                final_tags = [-100] * len(input_ids)

                tokens_info = []
                for idx, lbl in enumerate(chunk_labels):
                    if lbl is not None and lbl in self.label2id:
                        tokens_info.append({'idx': idx, 'label': lbl})

                if not tokens_info:
                    continue

                groups = []
                if len(tokens_info) > 0:
                    curr_group = [tokens_info[0]]
                    for k in range(1, len(tokens_info)):
                        if tokens_info[k]['label'] == tokens_info[k-1]['label']:
                            curr_group.append(tokens_info[k])
                        else:
                            groups.append(curr_group)
                            curr_group = [tokens_info[k]]
                    groups.append(curr_group)

                for grp in groups:
                    l_str = grp[0]['label']
                    indices = [t['idx'] for t in grp]

                    if len(indices) == 1:
                        final_tags[indices[0]] = self.tag2id[f"B-{l_str}"]
                    else:
                        final_tags[indices[0]] = self.tag2id[f"B-{l_str}"]
                        final_tags[indices[-1]] = self.tag2id[f"L-{l_str}"]
                        for mid_idx in indices[1:-1]:
                            final_tags[mid_idx] = self.tag2id[f"I-{l_str}"]

                self.samples.append({
                    'input_ids': torch.tensor(input_ids, dtype=torch.long),
                    'attention_mask': torch.tensor(mask, dtype=torch.long),
                    'labels': torch.tensor(final_tags, dtype=torch.long),
                    'sent_ids': torch.tensor(sent_ids, dtype=torch.long)
                })

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

In [13]:
def apply_correction(pred_ids, sent_ids, id2tag):

    sent_votes = collections.defaultdict(lambda: collections.defaultdict(int))

    for tag_idx, s_id in zip(pred_ids, sent_ids):
        if s_id == -1: continue # Пропускаем паддинг и спецтокены

        tag_str = id2tag.get(tag_idx, None)
        if tag_str:
            # tag_str типа "B-History", берем "History"
            clean_label = tag_str.split('-', 1)[1]
            sent_votes[s_id][clean_label] += 1

    corrected_sequence = []
    if not sent_votes:
        return []

    sorted_sids = sorted(sent_votes.keys())
    for sid in sorted_sids:
        votes = sent_votes[sid]
        # Берем класс с макс. голосами
        winner = max(votes, key=votes.get)
        corrected_sequence.append(winner)

    return corrected_sequence

def correct_bil_sequence(tag_ids, id2tag):

    if tag_ids is None or len(tag_ids) == 0:
        return tag_ids

    tag_ids_list = tag_ids.tolist() if isinstance(tag_ids, np.ndarray) else list(tag_ids)
    corrected = tag_ids_list.copy()
    valid_indices = [i for i, tag_id in enumerate(corrected) if tag_id in id2tag]

    if not valid_indices:
        return np.array(corrected) if isinstance(tag_ids, np.ndarray) else corrected

    tag_name_to_id = {v: k for k, v in id2tag.items()}

    def get_tag_id(tag_name):
        return tag_name_to_id.get(tag_name, -100)

    for j in range(len(valid_indices)):
        idx = valid_indices[j]  
        current_tag_id = corrected[idx]
        current_tag_name = id2tag.get(current_tag_id, "")
        
        if not current_tag_name:
            continue

        if "-" in current_tag_name:
            prefix, current_class = current_tag_name.split("-", 1)
        else:
            prefix, current_class = "", current_tag_name

        if j == 0:
            if prefix != "B":
                corrected[idx] = get_tag_id(f"B-{current_class}")

        elif j == len(valid_indices) - 1:
            prev_idx = valid_indices[j-1]
            prev_tag_name = id2tag.get(corrected[prev_idx], "")

            if "-" in prev_tag_name:
                prev_prefix, prev_class = prev_tag_name.split("-", 1)
                if prev_class == current_class:
                    if prefix != "L":
                        corrected[idx] = get_tag_id(f"L-{current_class}")
                else:
                    if prefix != "B":
                        corrected[idx] = get_tag_id(f"B-{current_class}")

        else:
            next_idx = valid_indices[j+1]
            next_tag_name = id2tag.get(corrected[next_idx], "")

            if "-" in next_tag_name:
                next_prefix, next_class = next_tag_name.split("-", 1)
            else:
                next_prefix, next_class = "", next_tag_name

            prev_idx = valid_indices[j-1]
            prev_tag_name = id2tag.get(corrected[prev_idx], "")

            if "-" in prev_tag_name:
                prev_prefix, prev_class = prev_tag_name.split("-", 1)
            else:
                prev_prefix, prev_class = "", prev_tag_name

            if next_class != current_class:
                if prefix != "L":
                    corrected[idx] = get_tag_id(f"L-{current_class}")
                if next_prefix != "B":
                    corrected[next_idx] = get_tag_id(f"B-{next_class}")

            elif prev_class != current_class:
                if prefix != "B":
                    corrected[idx] = get_tag_id(f"B-{current_class}")

    if len(valid_indices) == 1:
        idx = valid_indices[0]
        tag_name = id2tag.get(corrected[idx], "")
        if "-" in tag_name:
            prefix, cls = tag_name.split("-", 1)
            if prefix not in ["B", "L"]:
                corrected[idx] = get_tag_id(f"B-{cls}")

    return np.array(corrected) if isinstance(tag_ids, np.ndarray) else corrected

def get_tag_id_by_name(tag_name, id2tag):
    reverse_dict = {v: k for k, v in id2tag.items()}
    return reverse_dict.get(tag_name, -100)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, mask) # [Batch, Seq, Tags]

        # Flatten
        active_logits = logits.view(-1, logits.shape[-1])
        active_labels = labels.view(-1)

        loss = criterion(active_logits, active_labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, dataset, device):
    model.eval()

    all_f1_preds = []
    all_f1_labels = []
    pk_scores = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            sent_ids_batch = batch['sent_ids'].cpu().numpy()

            logits = model(input_ids, mask)
            preds = torch.argmax(logits, dim=2)

            preds_np = preds.cpu().numpy()
            labels_np = labels.cpu().numpy()
            sent_ids_np = sent_ids_batch

            for i in range(len(preds_np)):
                valid_mask = labels_np[i] != -100
                p_seq = preds_np[i][valid_mask]
                l_seq = labels_np[i][valid_mask]
                s_seq = sent_ids_np[i][valid_mask]

                p_seq = correct_bil_sequence(p_seq, dataset.id2tag)

                pred_tags = [dataset.id2tag[tid] for tid in p_seq]
                true_tags = [dataset.id2tag[tid] for tid in l_seq]

                if len(p_seq) == 0:
                    continue

                all_f1_preds.extend(p_seq)
                all_f1_labels.extend(l_seq)

                # Восстановление тем предложений
                corrected_pred = apply_correction(p_seq, s_seq, dataset.id2tag)
                corrected_true = apply_correction(l_seq, s_seq, dataset.id2tag)

                # Преобразуем темы в границы (1 если тема меняется, иначе 0)
                def to_boundaries(topics):
                    boundaries = []
                    for k in range(len(topics) - 1):
                        is_diff = 1 if topics[k] != topics[k+1] else 0
                        boundaries.append(is_diff)
                    if not boundaries:
                        return [0]
                    return boundaries

                ref_b = to_boundaries(corrected_true)
                hyp_b = to_boundaries(corrected_pred)

                if len(ref_b) == len(hyp_b) and len(ref_b) > 0:
                    pk = calculate_pk(ref_b, hyp_b)
                    pk_scores.append(pk)

    macro_f1 = f1_score(all_f1_labels, all_f1_preds, average='micro')
    mean_pk = np.mean(pk_scores) if pk_scores else 0.0

    print(f"\nFinal metrics:")
    print(f"Token-level F1: {macro_f1:.4f}")
    print(f"Sentence-level Pk: {mean_pk:.4f}")

    return macro_f1, mean_pk

In [15]:
# Параметры
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 7
LR = 0.00005

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
dataset_disease = load_dataset('json', data_files={
    'train': '/kaggle/input/disease/wikisection_en_disease_train.json',
    'test': '/kaggle/input/disease/wikisection_en_disease_test.json',
    'validation': '/kaggle/input/disease/wikisection_en_disease_validation.json'
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [20]:
print("Extracting unique labels...")
unique_labels = set()
for ann_list in tqdm(dataset_disease['train']['annotations']):
    for ann in ann_list:
        unique_labels.add(ann['sectionLabel'])

label2id = {l: i for i, l in enumerate(sorted(list(unique_labels)))}
print(f"Found {len(label2id)} classes: {list(label2id.keys())[:5]}...")

Extracting unique labels...


100%|██████████| 2513/2513 [00:00<00:00, 10422.72it/s]

Found 27 classes: ['disease.cause', 'disease.classification', 'disease.complication', 'disease.culture', 'disease.diagnosis']...


In [21]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [22]:
train_ds = WikiSectionDataset(dataset_disease['train'], tokenizer, label2id, max_len=MAX_LEN)
val_ds = WikiSectionDataset(dataset_disease['validation'], tokenizer, label2id, max_len=MAX_LEN)
test_ds = WikiSectionDataset(dataset_disease['test'], tokenizer, label2id, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

Preprocessing data and generating BIL tags...


100%|██████████| 2513/2513 [00:31<00:00, 78.81it/s] 


Preprocessing data and generating BIL tags...


100%|██████████| 359/359 [00:04<00:00, 74.74it/s]


Preprocessing data and generating BIL tags...


100%|██████████| 718/718 [00:09<00:00, 77.60it/s] 


In [23]:
num_tags = len(train_ds.tag2id)
print(f"Number of BIL tags: {num_tags}")

model = TATSSegmenter(MODEL_NAME, num_tags).to(device)

Number of BIL tags: 81


2026-01-16 11:14:22.904468: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768562063.087603      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768562063.141786      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768562063.553641      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768562063.553682      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768562063.553685      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [236]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = FocalLoss(gamma=2.0, ignore_index=-100)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [237]:
patience = 3 
best_pk = 100.0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch + 1}/{EPOCHS} ===")
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    print(f"Train Loss: {avg_loss:.4f}")

    f1, pk = evaluate(model, val_loader, val_ds, device)

    if pk < best_pk:
        best_pk = pk
        epochs_without_improvement = 0
        torch.save(model.state_dict(), f'/kaggle/working/model_perceptrone_2_{epoch}.pth')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping: нет улучшения качества сегментации более {patience} эпох подряд.")
            break

print("\nTraining complete!")


=== Epoch 1/7 ===


Training: 100%|██████████| 1936/1936 [26:07<00:00,  1.23it/s]


Train Loss: 1.1430


Evaluating: 100%|██████████| 277/277 [01:11<00:00,  3.90it/s]



Final metrics:
Token-level F1: 0.5253
Sentence-level Pk: 0.2193

=== Epoch 2/7 ===


...


Final metrics:
Token-level F1: 0.5474
Sentence-level Pk: 0.2120

=== Epoch 6/7 ===


Training: 100%|██████████| 1936/1936 [26:06<00:00,  1.24it/s]


Train Loss: 0.0517


Evaluating: 100%|██████████| 277/277 [01:11<00:00,  3.88it/s]



Final metrics:
Token-level F1: 0.5395
Sentence-level Pk: 0.2223
Early stopping: нет улучшения качества сегментации более 3 эпох подряд.

Training complete!


In [27]:
model_final_params_path = '/kaggle/input/perceprtone/pytorch/default/1/model_perceptrone_2_2.pth'

In [28]:
model = TATSSegmenter(MODEL_NAME, num_tags)

model.load_state_dict(torch.load(model_final_params_path, map_location=torch.device('cpu')))
model.to(device)
f1_test, pk_test = evaluate(model, test_loader, test_ds, device)

Evaluating: 100%|██████████| 525/525 [02:32<00:00,  3.45it/s]



Final metrics:
Token-level F1: 0.5425
Sentence-level Pk: 0.1987


In [247]:
f1_test

0.5424731944527826

In [248]:
pk_test

np.float64(0.1986510833427133)

In [38]:
def prediction_example(model, loader, dataset, device):
    model.eval()
    with torch.no_grad():
        data_iter = iter(loader)
        batch = next(data_iter)
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        sent_ids_batch = batch['sent_ids'].cpu().numpy()

        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=2)

        preds_np = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()
        sent_ids_np = sent_ids_batch

        for i in range(len(preds_np)):
            valid_mask = labels_np[i] != -100
            p_seq = preds_np[i][valid_mask]
            l_seq = labels_np[i][valid_mask]
            s_seq = sent_ids_np[i][valid_mask]

            p_seq = correct_bil_sequence(p_seq, dataset.id2tag)

            pred_tags = [dataset.id2tag[tid] for tid in p_seq]
            true_tags = [dataset.id2tag[tid] for tid in l_seq]

            print("Predicted tags:", pred_tags)
            print("True tags     :", true_tags)

            corrected_pred = apply_correction(p_seq, s_seq, dataset.id2tag)
            corrected_true = apply_correction(l_seq, s_seq, dataset.id2tag)

In [39]:
prediction_example(model, test_loader, test_ds, device)

Predicted tags: ['B-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.symptom', 'I-disease.sympto